# MLP digit classifier for VSDSquadron PRO
```
MLP handwritten-digit classifier for the VSDSquadron PRO (SiFive FE310)
=======================================================================
Train in Colab -> quantize to int8 -> export C headers -> run on the board.

Pipeline
  1. Load MNIST (28x28) and downscale every image to 16x16 (256 pixels).
  2. Train an MLP: 256 -> 64 (ReLU) -> 32 (ReLU) -> 10.
  3. Quantize for the MCU (no FPU on rv32imac, so keep the heavy math integer):
       - weights      : int8, one float scale per layer        (flash)
       - activations  : uint8, one float scale per hidden layer (RAM, tiny)
       - inputs       : raw uint8 pixels 0..255 (the 1/255 input scaling is
                        folded into layer-1's scale, so no pre-processing on MCU)
       - biases       : float
     Each neuron = int32 sum of (int8 weight * uint8 input), then ONE float
     multiply-add + ReLU. ~18.8k integer MACs and ~106 float ops per image.
  4. Re-run the model in numpy using exactly the arithmetic the C code uses,
     so the accuracy printed here is the accuracy the board should give.
  5. Export:  mlp_model_16x16.h   (weights/scales/biases, all const -> flash)
              mnist_test_16x16.h  (test images as uint8 + labels, const -> flash)
```

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
from PIL import Image
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# -------------------------------------------------------------------------
# 0. CONFIGURATION
# -------------------------------------------------------------------------
DATASET         = "mnist"   # "mnist" (needs internet, e.g. Colab) or "digits"
                            # (sklearn's built-in 8x8 digits upscaled to 16x16,
                            #  works offline - handy for a quick smoke test)
IMG_SIZE        = 16        # images are resized to IMG_SIZE x IMG_SIZE
HIDDEN_LAYERS   = (64, 32)  # hidden layer sizes
NUM_TEST_EXPORT = 10        # how many test images to put in the C header
RANDOM_STATE    = 0         # reproducible training

NUM_FEATURES = IMG_SIZE * IMG_SIZE

In [ ]:
# -------------------------------------------------------------------------
# 1. DATA LOADING & RESIZING
# -------------------------------------------------------------------------
def resize_images(images, size):
    """Resize a stack of 2-D uint8 images to size x size.
    BOX filter = area averaging, a good choice for shrinking images.
    The SAME function must be used for any image you later send to the board."""
    out = np.empty((len(images), size, size), dtype=np.uint8)
    for k, img in enumerate(images):
        out[k] = np.asarray(Image.fromarray(img).resize((size, size), Image.BOX))
    return out


if DATASET == "mnist":
    from keras.datasets import mnist
    (X_train_img, y_train), (X_test_img, y_test) = mnist.load_data()   # uint8 0..255
else:
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    d = load_digits()
    imgs = np.clip(d.images * 255.0 / 16.0, 0, 255).astype(np.uint8)  # 0..16 -> 0..255
    X_train_img, X_test_img, y_train, y_test = train_test_split(
        imgs, d.target, test_size=0.2, random_state=42)

print("Original image shape:", X_train_img.shape[1:])

X_train_u8 = resize_images(X_train_img, IMG_SIZE).reshape(-1, NUM_FEATURES)
X_test_u8  = resize_images(X_test_img,  IMG_SIZE).reshape(-1, NUM_FEATURES)
print("Resized train/test  :", X_train_u8.shape, X_test_u8.shape)

# The network is trained on pixels scaled to 0..1 (simple and MCU-friendly:
# the board just uses the raw 0..255 byte and the 1/255 lives in a constant).
X_train = X_train_u8.astype(np.float32) / 255.0
X_test  = X_test_u8.astype(np.float32)  / 255.0

Original image shape: (28, 28)
Resized train/test  : (60000, 256) (10000, 256)


In [ ]:
# -------------------------------------------------------------------------
# 2. TRAIN THE MLP
# -------------------------------------------------------------------------
mlp = MLPClassifier(
    hidden_layer_sizes=HIDDEN_LAYERS,
    activation='relu',          # ReLU = max(0, x): cheap on an MCU
    solver='adam',
    max_iter=100,               # epochs; early stopping usually ends sooner
    early_stopping=True,        # hold out 10% of train data, stop when it plateaus
    random_state=RANDOM_STATE,
)
mlp.fit(X_train, y_train)

y_pred_float = mlp.predict(X_test)
print(f"\nFloat model test accuracy : {accuracy_score(y_test, y_pred_float):.4f}")

n_params = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print("Trainable parameters      :", n_params)
print("Layer shapes (in, out)    :", [w.shape for w in mlp.coefs_])


Float model test accuracy : 0.9727
Trainable parameters      : 18858
Layer shapes (in, out)    : [(256, 64), (64, 32), (32, 10)]


In [ ]:
# -------------------------------------------------------------------------
# 3. QUANTIZATION
# -------------------------------------------------------------------------
# sklearn stores coefs_[l] with shape (inputs, outputs). For the C code we
# transpose to (outputs, inputs) so each neuron's weights are contiguous.
W = [w.T.astype(np.float32) for w in mlp.coefs_]        # [(64,256), (32,64), (10,32)]
B = [b.astype(np.float32)   for b in mlp.intercepts_]   # [(64,), (32,), (10,)]
n_layers = len(W)

# 3a. Weights -> int8, symmetric, one scale per layer: w_real = w_q * w_scale
w_scale = [np.abs(w).max() / 127.0 for w in W]
W_q = [np.clip(np.round(w / s), -127, 127).astype(np.int8) for w, s in zip(W, w_scale)]

# 3b. Hidden activations -> uint8: a_real = a_q * a_scale  (ReLU output >= 0)
#     Calibrate the range on training data. A high percentile instead of the
#     max ignores rare outliers so normal values keep more resolution.
def relu(x):
    return np.maximum(x, 0.0)

a = X_train
act_scale = []
for l in range(n_layers - 1):                       # hidden layers only
    a = relu(a @ W[l].T + B[l])
    act_scale.append(float(np.percentile(a, 99.99)) / 255.0 or 1.0)

# 3c. Input "scale": raw pixel byte p means p/255 -> input scale = 1/255
in_scale = 1.0 / 255.0

# 3d. Per-layer constants used by the C code (all float32):
#     layer 0: real_sum = acc * (w_scale0 * in_scale)
#     layer l: real_sum = acc * (w_scale_l * act_scale_{l-1})
#     hidden : a_q = clamp(round(relu(real_sum + bias) / act_scale_l), 0, 255)
#     output : score = real_sum + bias  (argmax -> predicted digit)
prev_scale = [in_scale] + act_scale
mult     = [np.float32(w_scale[l] * prev_scale[l]) for l in range(n_layers)]
inv_act  = [np.float32(1.0 / s) for s in act_scale]

In [ ]:
# -------------------------------------------------------------------------
# 4. BIT-ACCURATE SIMULATION OF THE C INFERENCE
# -------------------------------------------------------------------------
def predict_like_mcu(x_u8):
    """Integer MACs + float32 requantization, exactly as in mlp_inference.c."""
    a_q = x_u8.astype(np.int32)                                 # (N, 256)
    for l in range(n_layers):
        acc = a_q @ W_q[l].T.astype(np.int32)                   # int32 accumulators
        real = acc.astype(np.float32) * mult[l] + B[l]          # float32
        if l < n_layers - 1:                                    # hidden layer
            real = np.maximum(real, np.float32(0))
            a_q = np.minimum(np.floor(real * inv_act[l] + np.float32(0.5)), 255).astype(np.int32)
        else:                                                   # output layer
            return np.argmax(real, axis=1)

y_pred_q = predict_like_mcu(X_test_u8)
print(f"Quantized (MCU) accuracy  : {accuracy_score(y_test, y_pred_q):.4f}")
print(classification_report(y_test, y_pred_q, digits=3))

Quantized (MCU) accuracy  : 0.9723
              precision    recall  f1-score   support

           0      0.977     0.985     0.981       980
           1      0.987     0.990     0.989      1135
           2      0.978     0.973     0.975      1032
           3      0.961     0.977     0.969      1010
           4      0.971     0.969     0.970       982
           5      0.973     0.967     0.970       892
           6      0.973     0.979     0.976       958
           7      0.956     0.982     0.969      1028
           8      0.978     0.947     0.962       974
           9      0.969     0.949     0.959      1009

    accuracy                          0.972     10000
   macro avg      0.972     0.972     0.972     10000
weighted avg      0.972     0.972     0.972     10000



In [ ]:
# -------------------------------------------------------------------------
# 5. EXPORT C HEADERS
# -------------------------------------------------------------------------
def c_float(v):
    """Valid C float literal (0 -> 0.0f, 3 -> 3.0f, 1e-5 -> 1e-05f)."""
    s = f"{float(v):.9g}"
    if "." not in s and "e" not in s:
        s += ".0"
    return s + "f"

def c_array_1d(ctype, name, values, fmt=str, per_line=16):
    lines = []
    for k in range(0, len(values), per_line):
        lines.append("    " + ", ".join(fmt(v) for v in values[k:k + per_line]))
    return f"static const {ctype} {name}[{len(values)}] = {{\n" + ",\n".join(lines) + "\n};\n\n"

def c_array_2d(ctype, name, rows_name, cols_name, matrix, fmt=str):
    s = f"static const {ctype} {name}[{rows_name}][{cols_name}] = {{\n"
    for row in matrix:
        s += "    {" + ", ".join(fmt(v) for v in row) + "},\n"
    return s + "};\n\n"

sizes = [W[0].shape[1]] + [w.shape[0] for w in W]          # [256, 64, 32, 10]

with open("mlp_model_16x16.h", "w") as f:
    f.write("/* Auto-generated by mlp_mnist_16x16.py - do not edit by hand. */\n")
    f.write("#ifndef MLP_MODEL_16X16_H\n#define MLP_MODEL_16X16_H\n\n#include <stdint.h>\n\n")
    f.write(f"#define MLP_IN   {sizes[0]}   /* input pixels ({IMG_SIZE}x{IMG_SIZE}, uint8 0..255) */\n")
    f.write(f"#define MLP_H1   {sizes[1]}\n#define MLP_H2   {sizes[2]}\n")
    f.write(f"#define MLP_OUT  {sizes[3]}   /* digits 0..9 */\n\n")
    f.write("/* real_sum = int32_acc * MLP_Lx_MULT + bias[j]            */\n")
    f.write("/* hidden:  a_q = clamp(round(relu(real_sum) * MLP_Lx_INV_ACT), 0, 255) */\n")
    for l in range(n_layers):
        f.write(f"#define MLP_L{l+1}_MULT     {c_float(mult[l])}\n")
    for l in range(n_layers - 1):
        f.write(f"#define MLP_L{l+1}_INV_ACT  {c_float(inv_act[l])}\n")
    f.write("\n")
    dims = [("MLP_H1", "MLP_IN"), ("MLP_H2", "MLP_H1"), ("MLP_OUT", "MLP_H2")]
    for l in range(n_layers):
        f.write(f"/* Layer {l+1}: weights[out][in] int8 */\n")
        f.write(c_array_2d("int8_t", f"mlp_w{l+1}", dims[l][0], dims[l][1], W_q[l]))
        f.write(c_array_1d("float", f"mlp_b{l+1}", B[l], fmt=c_float, per_line=8))
    f.write("#endif /* MLP_MODEL_16X16_H */\n")
print("Exported mlp_model_16x16.h")

X_exp, y_exp = X_test_u8[:NUM_TEST_EXPORT], y_test[:NUM_TEST_EXPORT]
with open("mnist_test_16x16.h", "w") as f:
    f.write("/* Auto-generated by mlp_mnist_16x16.py - do not edit by hand. */\n")
    f.write("#ifndef MNIST_TEST_16X16_H\n#define MNIST_TEST_16X16_H\n\n#include <stdint.h>\n\n")
    f.write(f"#define NUM_TEST_IMAGES {len(X_exp)}\n")
    f.write(f"#define TEST_IMG_PIXELS {NUM_FEATURES}\n\n")
    f.write(c_array_2d("uint8_t", "test_images", "NUM_TEST_IMAGES", "TEST_IMG_PIXELS", X_exp))
    f.write(c_array_1d("uint8_t", "test_labels", [int(v) for v in y_exp]))
    f.write("#endif /* MNIST_TEST_16X16_H */\n")
print("Exported mnist_test_16x16.h")

# Expected board output for the exported images (compare with the UART log)
print("\nExpected predictions on the board:")
for i, (p, t) in enumerate(zip(predict_like_mcu(X_exp), y_exp)):
    print(f"Image {i}: Predicted = {p}, Actual = {t}")

flash = sum(w.size for w in W_q) + 4 * sum(b.size for b in B) + X_exp.size + len(y_exp)
print(f"\nApprox. flash for tables: {flash} bytes  (RAM for activations: "
      f"{sizes[1] + sizes[2]} bytes + {4 * sizes[3]} bytes scores)")

# Optional in Colab: download the headers
# from google.colab import files
# files.download("mlp_model_16x16.h"); files.download("mnist_test_16x16.h")

Exported mlp_model_16x16.h
Exported mnist_test_16x16.h

Expected predictions on the board:
Image 0: Predicted = 7, Actual = 7
Image 1: Predicted = 2, Actual = 2
Image 2: Predicted = 1, Actual = 1
Image 3: Predicted = 0, Actual = 0
Image 4: Predicted = 4, Actual = 4
Image 5: Predicted = 1, Actual = 1
Image 6: Predicted = 4, Actual = 4
Image 7: Predicted = 9, Actual = 9
Image 8: Predicted = 4, Actual = 5
Image 9: Predicted = 9, Actual = 9

Approx. flash for tables: 21746 bytes  (RAM for activations: 96 bytes + 40 bytes scores)


In [ ]:
from google.colab import files
files.download("mlp_model_16x16.h")
files.download("mnist_test_16x16.h")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!jupyter nbconvert "/content/drive/MyDrive/Colab Notebooks/MLP_MNIST_16x16.ipynb" --to html

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/MLP_MNIST_16x16.ipynb to html
[NbConvertApp] Writing 339795 bytes to /content/drive/MyDrive/Colab Notebooks/MLP_MNIST_16x16.html
